In [ ]:
from datasets import load_dataset
for cfg in ["corpus", "queries", "default"]:
    ds = load_dataset("mteb/fiqa", cfg)
    print(cfg, {k: (v.num_rows, v.column_names) for k, v in ds.items()})

c:\Users\ASUS\Documents\U\Semestre 7\Recuperacion de la informacion\Proyecto\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\ASUS\Documents\U\Semestre 7\Recuperacion de la informacion\Proyecto\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ASUS\.cache\huggingface\hub\datasets--mteb--fiqa. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to acti

In [ ]:
python -m scripts.crear_indice
python -m scripts.crear_embeddings

In [ ]:
import config
from src import almacen_vectores
from src.data_loader import cargar_json
from src.indice_invertido import IndiceInvertido
from src.modelos.modelo_binario import ModeloJaccard
from src.modelos.tfidf_coseno import ModeloTfidfCoseno
from src.modelos.bm25 import ModeloBM25
from src.modelos.semantico import GeneradorEmbeddings, ModeloSemantico

indice = IndiceInvertido.cargar(config.RUTA_INDICE_INVERTIDO)
faiss_idx, doc_ids = almacen_vectores.cargar(config.RUTA_FAISS, config.RUTA_DOC_IDS)
modelos = [ModeloJaccard(indice), ModeloTfidfCoseno(indice), ModeloBM25(indice),
           ModeloSemantico(GeneradorEmbeddings(), faiss_idx, doc_ids)]

consultas, qrels = cargar_json(config.RUTA_CONSULTAS), cargar_json(config.RUTA_QRELS)
qid = next(iter(consultas))
print(consultas[qid], "| relevantes:", list(qrels[qid]))
for m in modelos:
    print(m.nombre, [d for d, _ in m.buscar(consultas[qid], 5)])